# 🛡️ AI-Driven Phishing Email Detection Using NLP & Machine Learning

This notebook presents an end-to-end, production-grade Natural Language Processing (NLP) pipeline for detecting phishing emails.

### 🌟 Key Enhancements & Best Practices:
1. **Zero Data Leakage**: Stratified train/test split performed *before* TF-IDF vectorization and feature scaling.
2. **Domain-Specific Feature Extraction**: Extraction of phishing signals including URL counts, IP-based URLs, urgency keyword counts, uppercase character ratios, and punctuation intensity.
3. **Optimized Sparse Feature Scaling**: `MaxAbsScaler` scales metadata features to $[0, 1]$ while preserving sparse matrix efficiency and `MultinomialNB` compatibility.
4. **N-Gram Vectorization**: TF-IDF vectorizer with unigram and bigram extraction (`ngram_range=(1, 2)`) and sublinear TF scaling.
5. **Comprehensive Model Benchmarking**: 5 classifiers evaluated on Accuracy, Precision, Recall, F1-Score, and ROC-AUC.
6. **Production Inference Pipeline**: Live prediction function returning predictions, risk scores, confidence levels, and flag breakdowns.


In [1]:
import os
import re
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import hstack, csr_matrix

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MaxAbsScaler
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve
)

import warnings
warnings.filterwarnings("ignore")

# Style & Random Seed Configuration
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.size"] = 10

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✅ All required libraries imported successfully.")


✅ All required libraries imported successfully.


In [2]:
# Smart Dataset Path Detection (Local Workspace First, Google Colab Fallback)
LOCAL_DATASET_PATH = "phishing_email_cleaned.csv"
COLAB_DATASET_PATH = "/content/drive/MyDrive/phishing_email_cleaned.csv"

if os.path.exists(LOCAL_DATASET_PATH):
    dataset_path = LOCAL_DATASET_PATH
    print(f"📁 Found dataset in local directory: '{dataset_path}'")
elif os.path.exists(COLAB_DATASET_PATH):
    dataset_path = COLAB_DATASET_PATH
    print(f"📁 Found dataset in Google Drive: '{dataset_path}'")
else:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        dataset_path = COLAB_DATASET_PATH
    except ImportError:
        raise FileNotFoundError("❌ Dataset 'phishing_email_cleaned.csv' not found locally or in Google Drive.")

df_raw = pd.read_csv(dataset_path)

# Normalize column names & remove missing texts
if "text_combined" in df_raw.columns:
    df_raw = df_raw.rename(columns={"text_combined": "email_text"})

df_raw = df_raw.dropna(subset=["email_text"]).reset_index(drop=True)
df_raw["email_text"] = df_raw["email_text"].astype(str)

print("Dataset Overview:")
print(f"Total Emails: {df_raw.shape[0]:,}")
print(f"Total Columns: {df_raw.shape[1]}")
print("Class Distribution:")
print(df_raw["label"].value_counts().rename(index={0: "Legitimate (0)", 1: "Phishing (1)"}))
df_raw.head()


📁 Found dataset in local directory: 'phishing_email_cleaned.csv'
Dataset Overview:
Total Emails: 82,073
Total Columns: 2
Class Distribution:
label
Phishing (1)      42840
Legitimate (0)    39233
Name: count, dtype: int64


In [3]:
# Exploratory Data Analysis & Character Length Statistics
lengths = df_raw["email_text"].str.len()

print("Email Character-Length Distribution:")
print(lengths.describe())
print(f"Number of emails longer than 50,000 characters: {(lengths > 50000).sum()}")

print("-" * 60)
print("Sample Phishing Email (first 300 chars):")
print(df_raw[df_raw["label"] == 1]["email_text"].iloc[0][:300])
print("-" * 60)
print("Sample Legitimate Email (first 300 chars):")
print(df_raw[df_raw["label"] == 0]["email_text"].iloc[0][:300])
print("-" * 60)


Email Character-Length Distribution:
count    8.207300e+04
mean     1.290574e+03
std      1.553553e+04
min      1.000000e+01
25%      2.760000e+02
50%      5.580000e+02
75%      1.339000e+03
max      4.279526e+06
Name: email_text, dtype: float64
Number of emails longer than 50,000 characters: 56
------------------------------------------------------------
Sample Phishing Email (first 300 chars):
link dwl g 510 802 11 g wireless pci lan adapter 39 85 39 85 dwl g 510 high speed 2 4 ghz 802 11 g wireless pci lan adapter ieee 802 11 g standardupto 54 mbpsoperating frequency range 2 4 ghz ideal solution enabling wireless networking capabilities desktops pcs home office dwl g 510 visit http www c
------------------------------------------------------------
Sample Legitimate Email (first 300 chars):
hpl nom may 25 2001 see attached file hplno 525 xls hplno 525 xls
------------------------------------------------------------


In [4]:
# 1. Stratified Train / Test Split (BEFORE Any Preprocessing & Vectorization)
df_train, df_test = train_test_split(
    df_raw, test_size=0.20, random_state=RANDOM_STATE, stratify=df_raw["label"]
)

df_train = df_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

print("Train/Test Split Complete (Zero Data Leakage Guarantee)")
print(f"Training Set:   {len(df_train):,} samples ({len(df_train)/len(df_raw):.0%})")
print(f"Testing Set:    {len(df_test):,} samples ({len(df_test)/len(df_raw):.0%})")


Train/Test Split Complete (Zero Data Leakage Guarantee)
Training Set:   65,658 samples (80%)
Testing Set:    16,415 samples (20%)


In [5]:
# 2. Phishing Domain Feature Engineering
URGENT_WORDS = {
    "urgent", "immediately", "verify", "suspended", "action", "required",
    "click", "confirm", "expire", "expires", "alert", "blocked", "limited",
    "password", "security", "update", "account", "login", "bank", "risk"
}

def extract_metadata_features(df_subset):
    texts = df_subset["email_text"].astype(str).str.slice(0, 5000)
    
    url_count = texts.str.count(r"https?://|www\.")
    ip_url_count = texts.str.count(r"https?://\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}")
    texts_lower = texts.str.lower()
    urgent_word_count = texts_lower.apply(lambda text: sum(text.count(w) for w in URGENT_WORDS))
    capital_char_ratio = texts.apply(lambda text: sum(1 for c in text if c.isupper()) / (len(text) + 1))
    excl_count = texts.str.count(r"!")
    text_length = texts.str.len()
    word_count = texts.str.split().str.len()
    
    meta_df = pd.DataFrame({
        "url_count": url_count,
        "ip_url_count": ip_url_count,
        "urgent_word_count": urgent_word_count,
        "capital_char_ratio": capital_char_ratio,
        "excl_count": excl_count,
        "text_length": text_length,
        "word_count": word_count,
    }, index=df_subset.index)
    
    return meta_df

t0 = time.time()
meta_train = extract_metadata_features(df_train)
meta_test = extract_metadata_features(df_test)
meta_cols = list(meta_train.columns)

print(f"Extracted metadata features for Train & Test in {time.time()-t0:.2f} seconds.")
meta_train.head()


Extracted metadata features for Train & Test in 4.82 seconds.


In [6]:
# 3. Optimized Text Cleaning Function
MAX_CHARS = 5000

def clean_text(text):
    text = str(text)[:MAX_CHARS].lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

t0 = time.time()
clean_train = df_train["email_text"].apply(clean_text)
clean_test = df_test["email_text"].apply(clean_text)
print(f"Cleaned text for Train & Test in {time.time()-t0:.2f} seconds.")


Cleaned text for Train & Test in 3.24 seconds.


In [7]:
# 4. Vectorization (TF-IDF) & Feature Scaling (MaxAbsScaler)
t0 = time.time()

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=3,
    sublinear_tf=True
)

X_text_train = tfidf.fit_transform(clean_train)
X_text_test = tfidf.transform(clean_test)

scaler = MaxAbsScaler()
X_meta_train = csr_matrix(scaler.fit_transform(meta_train))
X_meta_test = csr_matrix(scaler.transform(meta_test))

X_train = hstack([X_text_train, X_meta_train]).tocsr()
X_test = hstack([X_text_test, X_meta_test]).tocsr()

y_train = df_train["label"].values
y_test = df_test["label"].values

feature_names = list(tfidf.get_feature_names_out()) + meta_cols

print(f"Vectorization & Scaling completed in {time.time()-t0:.2f} seconds.")
print(f"Final Feature Matrix Shape: Train {X_train.shape}, Test {X_test.shape}")


Vectorization & Scaling completed in 21.47 seconds.
Final Feature Matrix Shape: Train (65658, 5007), Test (16415, 5007)


In [8]:
# 5. Classifiers Initialization
models = {
    "Logistic Regression": LogisticRegression(
        C=2.0, max_iter=1000, random_state=RANDOM_STATE
    ),
    "Linear Support Vector Classifier": SGDClassifier(
        loss="log_loss", alpha=1e-5, max_iter=1000, random_state=RANDOM_STATE
    ),
    "Multinomial Naive Bayes": MultinomialNB(
        alpha=0.05
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, max_depth=25, random_state=RANDOM_STATE, n_jobs=-1
    ),
    "Neural Network (MLP)": MLPClassifier(
        hidden_layer_sizes=(32,), max_iter=150, early_stopping=True, random_state=RANDOM_STATE
    )
}

trained_models = {}
predictions = {}
predict_probas = {}
runtimes = {}

print("Model suite initialized.")


Model suite initialized.


In [9]:
# 6. Training & Prediction Benchmarking Loop
print("Training models on training set...")

for name, model in models.items():
    t0 = time.time()
    model.fit(X_train, y_train)
    t_train = time.time() - t0
    
    preds = model.predict(X_test)
    probas = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else preds
    
    trained_models[name] = model
    predictions[name] = preds
    predict_probas[name] = probas
    runtimes[name] = t_train
    
    print(f"Done: {name:32s} | Training Time: {t_train:6.2f}s")


Training models on training set...
Done: Logistic Regression              | Training Time:   0.30s
Done: Linear Support Vector Classifier | Training Time:   0.29s
Done: Multinomial Naive Bayes          | Training Time:   0.01s
Done: Random Forest                    | Training Time:   2.01s
Done: Neural Network (MLP)             | Training Time:   5.79s


In [10]:
# 7. Model Metrics Calculation & Comparative Analysis Table
results = []
for name in models.keys():
    preds = predictions[name]
    probas = predict_probas[name]
    
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds)
    rec = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    auc = roc_auc_score(y_test, probas)
    
    results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1,
        "ROC-AUC": auc,
        "Train Time (s)": runtimes[name]
    })

results_df = pd.DataFrame(results).sort_values("F1-Score", ascending=False).reset_index(drop=True)
results_df.to_csv("comparative_analysis_real_data.csv", index=False)

print("Model Benchmark Results:")
print(results_df.to_string(index=False))


Model Benchmark Results:
                           Model  Accuracy  Precision   Recall  F1-Score  ROC-AUC  Train Time (s)
            Neural Network (MLP)  0.984648   0.983713 0.986928  0.985318 0.998648        5.790480
             Logistic Regression  0.984161   0.981456 0.988329  0.984880 0.998415        0.304261
Linear Support Vector Classifier  0.983552   0.979986 0.988679  0.984313 0.998441        0.285346
                   Random Forest  0.955468   0.933798 0.984477  0.958468 0.991869        2.011191
         Multinomial Naive Bayes  0.950472   0.984143 0.919935  0.950956 0.994689        0.014567


In [11]:
# 8. Best Model Selection & Detailed Classification Report
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]

print(f"Best Performing Model: {best_model_name}")
print("=" * 60)
print(classification_report(
    y_test, predictions[best_model_name],
    target_names=["Legitimate (0)", "Phishing (1)"],
    digits=4
))


Best Performing Model: Neural Network (MLP)
                precision    recall  f1-score   support

Legitimate (0)     0.9857    0.9822    0.9839      7847
  Phishing (1)     0.9837    0.9869    0.9853      8568

      accuracy                         0.9846     16415
     macro avg     0.9847    0.9845    0.9846     16415
  weighted avg     0.9847    0.9846    0.9846     16415


In [12]:
# 9. Confusion Matrices Grid Visualization
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for idx, (name, preds) in enumerate(predictions.items()):
    ax = axes[idx]
    cm = confusion_matrix(y_test, preds)
    cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]
    
    annot = np.empty_like(cm, dtype=object)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            annot[i, j] = f"{cm[i, j]:,}\n({cm_norm[i, j]:.1%})"
            
    sns.heatmap(cm, annot=annot, fmt="", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["Legitimate", "Phishing"],
                yticklabels=["Legitimate", "Phishing"],
                annot_kws={"size": 11, "weight": "bold"})
    ax.set_title(name, fontsize=12, fontweight="bold")
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("Actual Label")

if len(models) < len(axes):
    fig.delaxes(axes[-1])

plt.tight_layout()
plt.savefig("confusion_matrices_real_data.png", dpi=300, bbox_inches="tight")
plt.close(fig)


In [13]:
# 10. Performance Visualizations: Metrics Bar Chart & ROC Curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

melted = results_df.melt(id_vars="Model", value_vars=["Accuracy", "Precision", "Recall", "F1-Score"],
                         var_name="Metric", value_name="Score")
sns.barplot(data=melted, x="Metric", y="Score", hue="Model", ax=ax1, palette="viridis")
ax1.set_ylim(0.90, 1.00)
ax1.set_title("Model Comparison Across NLP Evaluation Metrics", fontsize=13, fontweight="bold")
ax1.legend(bbox_to_anchor=(1.02, 1), loc="upper left")

for name, probas in predict_probas.items():
    fpr, tpr, _ = roc_curve(y_test, probas)
    auc_val = roc_auc_score(y_test, probas)
    ax2.plot(fpr, tpr, lw=2, label=f"{name} (AUC = {auc_val:.4f})")

ax2.plot([0, 1], [0, 1], color="gray", linestyle="--", lw=1.5)
ax2.set_xlabel("False Positive Rate (FPR)")
ax2.set_ylabel("True Positive Rate (TPR)")
ax2.set_title("Receiver Operating Characteristic (ROC) Curves", fontsize=13, fontweight="bold")
ax2.legend(loc="lower right")

plt.tight_layout()
plt.savefig("model_comparison_real_data.png", dpi=300, bbox_inches="tight")
plt.savefig("roc_curves_real_data.png", dpi=300, bbox_inches="tight")
plt.close(fig)


In [14]:
# 11. Feature Importance Analysis
if hasattr(best_model, "coef_"):
    importances = best_model.coef_[0]
elif hasattr(best_model, "feature_importances_"):
    importances = best_model.feature_importances_
else:
    lr_tmp = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    lr_tmp.fit(X_train, y_train)
    importances = lr_tmp.coef_[0]

top_n = 20
top_indices = np.argsort(np.abs(importances))[-top_n:]
top_features = [feature_names[i] for i in top_indices]
top_scores = importances[top_indices]

fig = plt.figure(figsize=(10, 7))
colors = ["crimson" if score > 0 else "navy" for score in top_scores]
plt.barh(top_features, top_scores, color=colors, alpha=0.85)
plt.xlabel("Feature Coefficient / Importance Impact", fontsize=11, fontweight="bold")
plt.title(f"Top {top_n} Most Influential Features ({best_model_name})", fontsize=13, fontweight="bold")
plt.axvline(0, color="black", linestyle="--", linewidth=0.8)
plt.tight_layout()
plt.savefig("feature_importance_real_data.png", dpi=300, bbox_inches="tight")
plt.close(fig)


In [15]:
# 12. Production Inference Function
def predict_new_email(email_text, model=None, model_name=None):
    if model is None:
        model_name = model_name or best_model_name
        model = trained_models[model_name]

    cleaned = clean_text(email_text)
    single_df = pd.DataFrame({"email_text": [email_text]})
    meta_df = extract_metadata_features(single_df)
    
    text_vec = tfidf.transform([cleaned])
    meta_vec = csr_matrix(scaler.transform(meta_df))
    combined_vec = hstack([text_vec, meta_vec]).tocsr()

    pred = model.predict(combined_vec)[0]
    proba_phishing = model.predict_proba(combined_vec)[0][1] if hasattr(model, "predict_proba") else float(pred)
    
    label = "PHISHING" if pred == 1 else "LEGITIMATE"
    confidence = proba_phishing if pred == 1 else (1 - proba_phishing)

    return {
        "Prediction": label,
        "Phishing Risk Score": f"{proba_phishing:.2%}",
        "Confidence": f"{confidence:.2%}",
        "Extracted Signals": {
            "URL Count": int(meta_df["url_count"].iloc[0]),
            "IP URL Count": int(meta_df["ip_url_count"].iloc[0]),
            "Urgent Words Count": int(meta_df["urgent_word_count"].iloc[0]),
            "Capital Char Ratio": f"{meta_df['capital_char_ratio'].iloc[0]:.1%}",
            "Exclamation Marks": int(meta_df["excl_count"].iloc[0]),
            "Total Characters": int(meta_df["text_length"].iloc[0])
        }
    }

sample_phishing = (
    "URGENT ATTENTION REQUIRED! Your online banking account access has been suspended due to suspicious activity. "
    "Click http://192.168.1.100/verify-identity immediately to log in and confirm your security credentials or your account will be deleted."
)

sample_legitimate = (
    "Hi Sarah, attached are the project roadmap updates for Q3. Please review the slides when you get a chance "
    "and let me know if you have any feedback before our Friday meeting. Thanks!"
)

import pprint
print("Live Test Case 1 (Suspicious Phishing Email):")
pprint.pprint(predict_new_email(sample_phishing))
print()
print("Live Test Case 2 (Legitimate Business Email):")
pprint.pprint(predict_new_email(sample_legitimate))


Live Test Case 1 (Suspicious Phishing Email):
{'Confidence': '100.00%',
 'Extracted Signals': {'Capital Char Ratio': '10.2%',
                       'Exclamation Marks': 1,
                       'IP URL Count': 1,
                       'Total Characters': 244,
                       'URL Count': 1,
                       'Urgent Words Count': 11},
 'Phishing Risk Score': '100.00%',
 'Prediction': 'PHISHING'}

Live Test Case 2 (Legitimate Business Email):
{'Confidence': '99.54%',
 'Extracted Signals': {'Capital Char Ratio': '3.3%',
                       'Exclamation Marks': 1,
                       'IP URL Count': 0,
                       'Total Characters': 181,
                       'URL Count': 0,
                       'Urgent Words Count': 1},
 'Phishing Risk Score': '0.46%',
 'Prediction': 'LEGITIMATE'}


In [16]:
# 13. Pipeline Serialization & Model Export
pipeline_payload = {
    "model": best_model,
    "model_name": best_model_name,
    "tfidf_vectorizer": tfidf,
    "scaler": scaler,
    "meta_cols": meta_cols,
    "feature_names": feature_names,
    "best_score_f1": results_df.iloc[0]["F1-Score"]
}

export_filename = "phishing_detector_pipeline.pkl"
joblib.dump(pipeline_payload, export_filename)

print(f"Full Phishing Detection Pipeline successfully saved to '{export_filename}'.")
print(f"Included Model: {best_model_name} (F1 Score: {results_df.iloc[0]['F1-Score']:.4%})")


Full Phishing Detection Pipeline successfully saved to 'phishing_detector_pipeline.pkl'.
Included Model: Neural Network (MLP) (F1 Score: 98.5318%)
